# Tokenizer Comparison — Llama-2, OPT-350M, Pythia-410M

Loads **tokenizers only** (no model weights) for the three models used across the
distillation experiments, then shows how each one segments the same input text.
Useful for sanity-checking vocabulary / subword differences between teacher
(Llama-2) and the two student candidates (OPT-350M, Pythia-410M).

In [ ]:
import os

HOME = '/workspace'
print(HOME)

# Tokenizer paths — adjust if your local model directory is different.
LLAMA_TOKENIZER  = f"{HOME}/models/Llama-2-7b-chat-hf"
OPT_TOKENIZER    = f"{HOME}/Multi-Level-OT/EleutherAI/opt-350m"
PYTHIA_TOKENIZER = f"{HOME}/Multi-Level-OT/EleutherAI/pythia-410m"

TOKENIZER_PATHS = {
    "llama2": LLAMA_TOKENIZER,
    "opt":    OPT_TOKENIZER,
    "pythia": PYTHIA_TOKENIZER,
}

In [ ]:
# Imports — AutoTokenizer only, never AutoModel*
from transformers import AutoTokenizer

print("Imports OK")

In [ ]:
# Load tokenizers only — no full model is instantiated anywhere in this notebook.
tokenizers = {}
for name, path in TOKENIZER_PATHS.items():
    tok = AutoTokenizer.from_pretrained(path, use_fast=True)
    tokenizers[name] = tok
    print(f"[{name:7s}] loaded from {path}")
    print(f"          vocab_size={tok.vocab_size}  fast={tok.is_fast}  "
          f"pad={tok.pad_token!r}  eos={tok.eos_token!r}  bos={tok.bos_token!r}")

---
## Input

In [ ]:
INPUT_TEXT = "The quick brown fox jumps over the lazy dog."
print(repr(INPUT_TEXT))

---
## Tokenization results

For each tokenizer: token count, the token strings, their ids, and the
round-trip decode (special tokens excluded so the three are directly comparable).

In [ ]:
def show_tokenization(text: str, tokenizer, name: str) -> None:
    ids = tokenizer.encode(text, add_special_tokens=False)
    toks = tokenizer.convert_ids_to_tokens(ids)
    print(f"--- {name} ---")
    print(f"  n_tokens : {len(ids)}")
    print(f"  tokens   : {toks}")
    print(f"  ids      : {ids}")
    print(f"  decoded  : {tokenizer.decode(ids)!r}")
    print()


for name, tok in tokenizers.items():
    show_tokenization(INPUT_TEXT, tok, name)

In [ ]:
# Side-by-side token-count summary
print(f"{'tokenizer':10s} {'n_tokens':>8s}")
for name, tok in tokenizers.items():
    n = len(tok.encode(INPUT_TEXT, add_special_tokens=False))
    print(f"{name:10s} {n:8d}")